# Chunking de manifestações longas
**Fernando Amorim Pontes Neto**

Comparamos (240, 0), (240, 60) e (400, 80), em caracteres. As duas primeiras configurações isolam o efeito do overlap. A terceira explora mais contexto por chunk. A base do enunciado descreve cinco textos com mais de 500 caracteres; o contexto menciona palavras, mas a especificação do dataset usa caracteres.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity
from ouvidoria_core import *
base = carregar_base('manifestacoes.json')
assert len(base) == 40, 'A análise do desafio requer exatamente 40 registros.'
textos = base.texto.tolist()
ids = base.id.tolist()
saida = Path('resultados'); saida.mkdir(exist_ok=True)
print(f'Base: {len(base)} registros; textos >500 caracteres: {(base.texto.str.len()>500).sum()}')
if Path('PROVENIENCIA.json').exists():
    display(Markdown('**Proveniência:** ' + json.loads(Path('PROVENIENCIA.json').read_text())['descricao']))
display(base.head())

/home/user/ouvidoria/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Base: 40 registros; textos >500 caracteres: 5


**Proveniência:** Base demonstrativa sintética criada para o exercício, com autorização do usuário. Não corresponde a manifestações reais nem à base original do professor. Os resultados e o gabarito se aplicam apenas a estes 40 exemplos.

,id,data,categoria_oficial,texto
0,M001,2026-09-01,infraestrutura,"A calçada da Rua das Flores, perto do número 1..."
1,M002,2026-09-01,educação,A Escola Municipal Esperança está sem professo...
2,M003,2026-09-02,infraestrutura,"Há um buraco enorme na Av. Brasil, a avenida p..."
3,M004,2026-09-02,segurança,Moradores relatam furtos de bicicletas junto a...
4,M005,2026-09-03,meio ambiente,Uma pilha de pneus usados foi abandonada no te...


In [2]:
longas=base.assign(tamanho=base.texto.str.len()).sort_values(['tamanho','id'],ascending=[False,True]).head(5)
display(longas[['id','tamanho','texto']])
assert (longas.tamanho>500).all(), 'As cinco maiores manifestações devem ter mais de 500 caracteres.'
configuracoes=[(240,0),(240,60),(400,80)]
resumos=[]; detalhes=[]
for tamanho,overlap in configuracoes:
    registros=[]
    for r in longas.itertuples():
        for chunk in dividir(r.texto,tamanho,overlap):
            registros.append({'id':r.id,**chunk})
    chunks=pd.DataFrame(registros)
    e=codificar(chunks.texto.tolist()); sim=e@e.T
    adj=[]; sobreposicoes=[]; intra=[]; inter=[]
    for i in range(len(chunks)):
        for j in range(i+1,len(chunks)):
            (intra if chunks.id.iloc[i]==chunks.id.iloc[j] else inter).append(float(sim[i,j]))
        if i+1<len(chunks) and chunks.id.iloc[i]==chunks.id.iloc[i+1]:
            adj.append(float(sim[i,i+1]))
            sobreposicoes.append(max(0,int(chunks.inicio.iloc[i]+chunks.caracteres.iloc[i]-chunks.inicio.iloc[i+1])))
    r={'chunk_size':tamanho,'chunk_overlap':overlap,'n_chunks':len(chunks),
       'cosseno_adjacente':float(np.mean(adj)) if adj else None,
       'overlap_efetivo_medio':float(np.mean(sobreposicoes)) if sobreposicoes else 0,
       'cosseno_mesma_manifestacao':float(np.mean(intra)) if intra else None,
       'cosseno_outras_manifestacoes':float(np.mean(inter)) if inter else None}
    resumos.append(r)
    chunks.to_csv(saida/f'chunks_{tamanho}_{overlap}.csv',index=False)
    display(Markdown(f'### Configuração {tamanho} e {overlap}'))
    display(chunks)
    coords=projetar(e,'PCA'); fig,ax=plt.subplots(figsize=(10,5))
    for ident in longas.id:
        mask=(chunks.id==ident).to_numpy()
        ax.scatter(coords[mask,0],coords[mask,1],label=ident)
    for i,xy in enumerate(coords): ax.annotate(f'{chunks.id.iloc[i]} C{chunks.chunk.iloc[i]}',xy,fontsize=7)
    ax.legend(); ax.set_title(f'PCA dos chunks — tamanho {tamanho}, overlap {overlap}')
    fig.tight_layout(); fig.savefig(saida/f'chunks_pca_{tamanho}_{overlap}.png',dpi=150); plt.show()
    exemplo=chunks[chunks.id==longas.id.iloc[0]].head(2)
    detalhes.append({'configuracao':[tamanho,overlap],'exemplo':exemplo.to_dict('records')})
display(pd.DataFrame(resumos).round(4))
(saida/'chunking_resumo.json').write_text(json.dumps({'configuracoes':resumos,'exemplos':detalhes},ensure_ascii=False,indent=2))

,id,tamanho,texto
15,M016,737,"Na Unidade de Saúde Vila Nova, o agendamento d..."
39,M040,736,"Na margem do açude do bairro Campo, apareceram..."
23,M024,733,"Na Escola Municipal Caminhos, a merenda tem si..."
9,M010,720,"Na Rua dos Ipês, entre os números 200 e 260, a..."
32,M033,707,"Na saída do terminal do bairro Central, o flux..."


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3516.50it/s]

### Configuração 240 e 0

,id,chunk,inicio,caracteres,texto
0,M016,1,0,127,"Na Unidade de Saúde Vila Nova, o agendamento d..."
1,M016,2,127,237,. Cheguei às seis horas em dois dias diferente...
2,M016,3,364,124,. Não estou reclamando da consulta com o médic...
3,M016,4,488,145,. Peço que a gestão informe uma alternativa de...
4,M016,5,633,104,. Também precisamos de informações claras no b...
5,M040,1,0,129,"Na margem do açude do bairro Campo, apareceram..."
6,M040,2,129,231,. O descarte fica próximo ao caminho usado pel...
7,M040,3,360,99,. Solicitamos recolhimento do material e fisca...
8,M040,4,459,145,. Também pedimos orientação sobre o local corr...
9,M040,5,604,132,. A limpeza precisa incluir a margem e o camin...


### Configuração 240 e 60

,id,chunk,inicio,caracteres,texto
0,M016,1,0,127,"Na Unidade de Saúde Vila Nova, o agendamento d..."
1,M016,2,127,237,. Cheguei às seis horas em dois dias diferente...
2,M016,3,364,124,. Não estou reclamando da consulta com o médic...
3,M016,4,488,145,. Peço que a gestão informe uma alternativa de...
4,M016,5,633,104,. Também precisamos de informações claras no b...
5,M040,1,0,129,"Na margem do açude do bairro Campo, apareceram..."
6,M040,2,129,231,. O descarte fica próximo ao caminho usado pel...
7,M040,3,360,99,. Solicitamos recolhimento do material e fisca...
8,M040,4,459,145,. Também pedimos orientação sobre o local corr...
9,M040,5,604,132,. A limpeza precisa incluir a margem e o camin...


### Configuração 400 e 80

,id,chunk,inicio,caracteres,texto
0,M016,1,0,364,"Na Unidade de Saúde Vila Nova, o agendamento d..."
1,M016,2,364,373,. Não estou reclamando da consulta com o médic...
2,M040,1,0,360,"Na margem do açude do bairro Campo, apareceram..."
3,M040,2,360,376,. Solicitamos recolhimento do material e fisca...
4,M024,1,0,344,"Na Escola Municipal Caminhos, a merenda tem si..."
5,M024,2,344,389,. Pedimos que a secretaria verifique a regular...
6,M010,1,0,350,"Na Rua dos Ipês, entre os números 200 e 260, a..."
7,M010,2,350,370,. Já registramos pedido de limpeza na semana a...
8,M033,1,0,362,"Na saída do terminal do bairro Central, o flux..."
9,M033,2,362,345,. Não se trata de solicitação de obra na calça...


,chunk_size,chunk_overlap,n_chunks,cosseno_adjacente,overlap_efetivo_medio,cosseno_mesma_manifestacao,cosseno_outras_manifestacoes
0,240,0,22,0.4493,0.0,0.4247,0.2586
1,240,60,22,0.4493,0.0,0.4247,0.2586
2,400,80,10,0.6070,0.0,0.6070,0.3759


3503

In [3]:
a,b,c=resumos
mudanca=b['cosseno_adjacente']-a['cosseno_adjacente']
display(Markdown(f"Mantendo chunk_size=240, acrescentar overlap=60 mudou o cosseno médio dos vizinhos em **{mudanca:+.3f}**. Parte da variação pode vir das palavras repetidas e não de melhor compreensão. O overlap efetivo médio foi **{b['overlap_efetivo_medio']:.1f} caracteres**, pois o separador recursivo respeita os limites disponíveis."))
for r in resumos:
    intra=r['cosseno_mesma_manifestacao']; inter=r['cosseno_outras_manifestacoes']
    display(Markdown(f"Em ({r['chunk_size']}, {r['chunk_overlap']}), a média de cosseno entre chunks da mesma manifestação é **{intra:.3f}**, contra **{inter:.3f}** entre manifestações. "+('Há proximidade relativa no espaço original.' if intra>inter else 'A origem comum não produziu maior proximidade média.')))
for detalhe in detalhes:
    display(Markdown(f"**Exemplo de limites — {detalhe['configuracao']}**"))
    for trecho in detalhe['exemplo']:
        print(f"{trecho['id']} / chunk {trecho['chunk']} / início {trecho['inicio']}: {trecho['texto']}")

Mantendo chunk_size=240, acrescentar overlap=60 mudou o cosseno médio dos vizinhos em **+0.000**. Parte da variação pode vir das palavras repetidas e não de melhor compreensão. O overlap efetivo médio foi **0.0 caracteres**, pois o separador recursivo respeita os limites disponíveis.

Em (240, 0), a média de cosseno entre chunks da mesma manifestação é **0.425**, contra **0.259** entre manifestações. Há proximidade relativa no espaço original.

Em (240, 60), a média de cosseno entre chunks da mesma manifestação é **0.425**, contra **0.259** entre manifestações. Há proximidade relativa no espaço original.

Em (400, 80), a média de cosseno entre chunks da mesma manifestação é **0.607**, contra **0.376** entre manifestações. Há proximidade relativa no espaço original.

**Exemplo de limites — [240, 0]**

M016 / chunk 1 / início 0: Na Unidade de Saúde Vila Nova, o agendamento de exames tem sido interrompido porque o sistema fica indisponível durante a manhã
M016 / chunk 2 / início 127: . Cheguei às seis horas em dois dias diferentes e, após esperar por horas, fui orientado a voltar sem receber protocolo. Outras pessoas na fila relataram a mesma dificuldade, inclusive moradores que dependem de transporte de outro bairro


**Exemplo de limites — [240, 60]**

M016 / chunk 1 / início 0: Na Unidade de Saúde Vila Nova, o agendamento de exames tem sido interrompido porque o sistema fica indisponível durante a manhã
M016 / chunk 2 / início 127: . Cheguei às seis horas em dois dias diferentes e, após esperar por horas, fui orientado a voltar sem receber protocolo. Outras pessoas na fila relataram a mesma dificuldade, inclusive moradores que dependem de transporte de outro bairro


**Exemplo de limites — [400, 80]**

M016 / chunk 1 / início 0: Na Unidade de Saúde Vila Nova, o agendamento de exames tem sido interrompido porque o sistema fica indisponível durante a manhã. Cheguei às seis horas em dois dias diferentes e, após esperar por horas, fui orientado a voltar sem receber protocolo. Outras pessoas na fila relataram a mesma dificuldade, inclusive moradores que dependem de transporte de outro bairro
M016 / chunk 2 / início 364: . Não estou reclamando da consulta com o médico, que ocorreu normalmente; o problema está na marcação dos exames solicitados. Peço que a gestão informe uma alternativa de agendamento quando houver falha do sistema e organize uma lista com os pacientes que já aguardavam. Também precisamos de informações claras no balcão para evitar deslocamentos repetidos sem atendimento.


## Interpretação e escolha
A configuração (400, 80) é a escolha inicial para a interface quando a denúncia descreve local, ocorrência e consequência em frases consecutivas: o trecho maior oferece mais espaço para essas informações permanecerem juntas. Os exemplos acima permitem conferir os limites reais; um score alto não basta para concluir que o sentido foi preservado. (240, 60) favorece recuperação de detalhes e custa mais vetores; (240, 0) não repete contexto e pode separar uma referência do objeto a que se refere. O overlap deve ser escolhido após inspecionar os trechos e avaliar consultas representativas. A PCA é apenas diagnóstico visual; as médias no espaço original são a evidência quantitativa de proximidade. Janelas semânticas podem ainda separar temas diferentes dentro de uma mesma denúncia.

## Referências
Material fornecido: app_chunks_embeddings.py. RecursiveCharacterTextSplitter: https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter. PCA e t-SNE: https://scikit-learn.org/stable/modules/manifold.html.